# Task 3: Soft MoE Optuna Search & Full Retraining

Hyperparameter optimization and full schedule retraining for the soft Mixture-of-Experts system as specified in the assignment.

### Mathematical Formulation
$$w = \mathrm{softmax}\!\left(\frac{g(\tilde{x})}{\tau}\right), \quad \hat{x} = w_c\,\tilde{x} + w_s\,D_{salt}(\tilde{x}) + w_b\,D_{blur}(\tilde{x}) + w_o\,D_{occlusion}(\tilde{x})$$
$$\mathcal{L}_{joint} = \lambda_1\,\mathcal{L}_{L1} + \lambda_2\,(1-\mathrm{SSIM}) + \lambda_3\,\mathcal{L}_{CE} + \lambda_4\,\mathcal{L}_{balance}$$
$$\mathcal{L}_{balance} = \sum_{k=1}^{4}\left(\bar{w}_k - \frac{1}{4}\right)^{2}$$

### Optuna Search Space Requirements
As required by the assignment, Optuna tunes:
1. **Joint Fine-Tuning Learning Rate** ($\text{finetune\_lr} \in [10^{-5}, 10^{-4}]$ log scale)
2. **Softmax Temperature** ($\tau \in [0.5, 2.0]$)
3. **Classification Loss Weight** ($\lambda_3 \in [0.01, 0.5]$)
4. **Routing Balance Regularizer Weight** ($\lambda_4 \in [10^{-3}, 10^{-1}]$ log scale)
5. **Reconstruction-Loss Weighting** ($\lambda_1 = \alpha, \lambda_2 = 1 - \alpha$ with $\alpha \in [0.5, 0.95]$)

### Retraining:
The winning configuration is retrained for `FULL_TRAIN_EPOCHS = 25` to produce `task3_soft_moe_final_best.pt` for deployment and benchmarking.

In [ ]:
# 1. Config
N_TRIALS = 15
OPTUNA_EPOCHS = 12
STUDY_NAME = "task3_soft_moe"
STORAGE_DB = "sqlite:///task3_study.db"
SEED = 42

In [ ]:
# 2. Imports & Setup
import sys
from pathlib import Path
import yaml
import torch
from torch.utils.data import DataLoader
import optuna

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path: sys.path.insert(0, str(RESEARCH_ROOT))
from constants import CONFIGS_ROOT
from src.device_utils import get_device
from src.datasets import PetDataset
from src.models_classifier import CorruptionClassifier, SoftMoERestorer
from src.models_dae import ConvDAE
from src.train_loop import train_one_epoch_moe, validate_moe, set_seed
from src.checkpoint_utils import load_checkpoint
set_seed(SEED)
device = get_device()
checkpoint_dir = RESEARCH_ROOT / "checkpoints"
gate_path = checkpoint_dir / "task2_classifier_best.pt"
expert_paths = {"salt": checkpoint_dir / "task2b_specialist_salt_best.pt",
                "blur": checkpoint_dir / "task2b_specialist_blur_best.pt",
                "occlusion": checkpoint_dir / "task2b_specialist_occlusion_best.pt"}
required = [gate_path, *expert_paths.values()]
missing = [str(path) for path in required if not path.exists()]
if missing: raise FileNotFoundError("Train Tasks 2a and 2b before MoE search: " + ", ".join(missing))


In [ ]:
# 3. Objective Function
train_dataset = PetDataset(mode="train", corruption_mode="all", return_label=True, seed=SEED)
val_dataset = PetDataset(mode="val", corruption_mode="all", return_label=True, seed=SEED)

def objective(trial: optuna.Trial) -> float:
    finetune_lr = trial.suggest_float("finetune_lr", 1e-5, 1e-4, log=True)
    temperature = trial.suggest_float("temperature", 0.5, 2.0)
    lambda3 = trial.suggest_float("lambda3", 0.01, 0.5)
    lambda4 = trial.suggest_float("lambda4", 1e-3, 1e-1, log=True)
    alpha = trial.suggest_float("alpha", 0.5, 0.95)
    lambdas = {"lambda1": alpha, "lambda2": 1-alpha,
               "lambda3": lambda3, "lambda4": lambda4}
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
    gate_state = torch.load(gate_path, map_location=device, weights_only=False)["model_state_dict"]
    gate = CorruptionClassifier(base_channels=gate_state["conv1.0.weight"].shape[0]).to(device)
    gate.load_state_dict(gate_state)
    specialists = []
    for path in expert_paths.values():
        state = torch.load(path, map_location=device, weights_only=False)["model_state_dict"]
        model = ConvDAE(base_channels=state["enc1.0.weight"].shape[0],
                        latent_dim=state["fc_z.weight"].shape[0]).to(device)
        model.load_state_dict(state)
        specialists.append(model)
    moe = SoftMoERestorer(gate, *specialists, temperature=temperature).to(device)
    optimizer = torch.optim.Adam(moe.parameters(), lr=finetune_lr)
    best_loss = float("inf")
    for epoch in range(1, OPTUNA_EPOCHS + 1):
        train_one_epoch_moe(moe, train_loader, optimizer, device, lambdas)
        metrics = validate_moe(moe, val_loader, device, lambdas)
        best_loss = min(best_loss, metrics["val_loss"])
        trial.set_user_attr(f"epoch_{epoch}_routing", metrics["avg_expert_weights"].tolist())
        trial.report(metrics["val_loss"], step=epoch)
        if trial.should_prune(): raise optuna.TrialPruned()
    return best_loss


In [ ]:
# 5. Retrain Winning Soft MoE on Full Schedule
FULL_TRAIN_EPOCHS = 25
WARMUP_EPOCHS = 5
p = study.best_params
print(f"\n--- Retraining Winning Soft MoE ({FULL_TRAIN_EPOCHS} Joint Epochs) ---")
print(p)

lambdas = {
    "lambda1": p["alpha"],
    "lambda2": 1.0 - p["alpha"],
    "lambda3": p["lambda3"],
    "lambda4": p["lambda4"]
}

# Initialize fresh gate and specialists from Task 2 checkpoints
gate_state = torch.load(gate_path, map_location=device, weights_only=False)["model_state_dict"]
gate = CorruptionClassifier(base_channels=gate_state["conv1.0.weight"].shape[0]).to(device)
gate.load_state_dict(gate_state)

sp_salt = ConvDAE(base_channels=torch.load(expert_paths["salt"], map_location=device, weights_only=False)["model_state_dict"]["enc1.0.weight"].shape[0],
                  latent_dim=torch.load(expert_paths["salt"], map_location=device, weights_only=False)["model_state_dict"]["fc_z.weight"].shape[0]).to(device)
sp_salt.load_state_dict(torch.load(expert_paths["salt"], map_location=device, weights_only=False)["model_state_dict"])

sp_blur = ConvDAE(base_channels=torch.load(expert_paths["blur"], map_location=device, weights_only=False)["model_state_dict"]["enc1.0.weight"].shape[0],
                  latent_dim=torch.load(expert_paths["blur"], map_location=device, weights_only=False)["model_state_dict"]["fc_z.weight"].shape[0]).to(device)
sp_blur.load_state_dict(torch.load(expert_paths["blur"], map_location=device, weights_only=False)["model_state_dict"])

sp_occl = ConvDAE(base_channels=torch.load(expert_paths["occlusion"], map_location=device, weights_only=False)["model_state_dict"]["enc1.0.weight"].shape[0],
                  latent_dim=torch.load(expert_paths["occlusion"], map_location=device, weights_only=False)["model_state_dict"]["fc_z.weight"].shape[0]).to(device)
sp_occl.load_state_dict(torch.load(expert_paths["occlusion"], map_location=device, weights_only=False)["model_state_dict"])

moe = SoftMoERestorer(gate, sp_salt, sp_blur, sp_occl, temperature=p["temperature"]).to(device)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

# Stage 1: Warmup
for sp in [moe.specialist_salt, moe.specialist_blur, moe.specialist_occlusion]:
    for param in sp.parameters(): param.requires_grad = False
for param in moe.gate.parameters(): param.requires_grad = True

opt_warmup = torch.optim.Adam(moe.gate.parameters(), lr=1e-3)
for ep in range(1, WARMUP_EPOCHS + 1):
    train_one_epoch_moe(moe, train_loader, opt_warmup, device, lambdas)

# Stage 2: Joint Finetuning
for param in moe.parameters(): param.requires_grad = True
opt_joint = torch.optim.Adam(moe.parameters(), lr=p["finetune_lr"])
best_val_loss = float("inf")

for epoch in range(1, FULL_TRAIN_EPOCHS + 1):
    tr = train_one_epoch_moe(moe, train_loader, opt_joint, device, lambdas)
    vl = validate_moe(moe, val_loader, device, lambdas)
    is_best = vl["val_loss"] < best_val_loss
    if is_best: best_val_loss = vl["val_loss"]
    ckpt = create_checkpoint_dict(
        moe, opt_joint, epoch, vl["val_loss"],
        optuna_trial_number=int(study.best_trial.number),
        random_seed=SEED, extra_metadata={**p, **lambdas}
    )
    save_checkpoint(
        checkpoint_dir=checkpoint_dir, prefix="task3_soft_moe_final",
        ckpt_dict=ckpt, is_best=is_best, task_name="task3",
        phase_name="final", device_name=device.type
    )
    print(f"Epoch [{epoch:02d}/{FULL_TRAIN_EPOCHS:02d}] Val Loss: {vl['val_loss']:.4f} (SSIM: {vl['val_ssim']:.4f}) {'*Best*' if is_best else ''}")

print(f"Final MoE Retraining Complete! Best Val Loss: {best_val_loss:.4f}")